# Module 3.3 — Notebook A: Evaluate the baseline

In this notebook you test whether Candlekeep's routing prototype is strong enough to move forward.

Candlekeep handles 20,000+ customer messages per month, with roughly 60 hours of manual processing per week and an average 4.2-hour first response time. If routing can be automated safely and affordably, the team can reduce manual effort and speed up response time.

The goal is not to prove perfection. The goal is to make one responsible decision — continue, improve and re-test, or stop — using evidence from a fixed offline evaluation set.

## What this notebook evaluates

Each step maps to a hypothesis from Module 3.1:

| Dimension | Hypothesis | Pass threshold |
|-----------|-----------|----------------|
| **Routing quality** | H2 — the model routes correctly often enough to reduce manual work | ≥85% department accuracy; misroute rate ≤10% |
| **Safety** | Confidence guardrail — high-confidence cases can be auto-routed safely | Unsafe auto-route rate ≤3% |
| **Cost** | H6 — inference cost is acceptable at Candlekeep volume | <$1,000/month at 20,000 messages |
| **Speed** | H7 — the model responds fast enough to improve first response time | p95 latency <5,000 ms |

You will compare a few candidate models on a sample, choose one to study further, then run a full evaluation on that model.

## Setup

Install dependencies and load helpers. Run the cell below, then move on to **Settings**.

- **Colab:** the cell installs from GitHub automatically.
- **Local:** replace the install line with `%pip install -q -e .` (from the repo root).

Keep your API key in **Secrets** (Colab) or a local **`.env`** file — avoid committing real keys.

In [ ]:
%pip install -q git+https://github.com/mnrozhkov/ai_leader.git
%matplotlib inline

import os

from dotenv import load_dotenv

from ai_leader import *

load_dotenv(override=False)

## Step 1 — Settings

Set your API key and experiment parameters below.

You can change the **dataset URL** (to use your own data) and **candidate models**. For model comparison we use **30%** of the rows so runs stay fast; the **full dataset** is used in Step 4.

### API key & parameters

This notebook calls the **Nebius inference API** (OpenAI-compatible).
Get your key at [Nebius AI Studio](https://studio.nebius.ai) → Settings → API Keys.
Store it as a Colab Secret named `NEBIUS_API_KEY`, or add it to a local `.env` file — do **not** paste a real key into the notebook.

**401 error?** Make sure you use a Nebius *inference* API key (not an OpenAI key). See [Nebius API authentication](https://docs.nebius.com/studio/api/authentication) for regional endpoint options.

In [ ]:
api_key = setup_api_key()

# Dataset — replace with your own CSV URL or local path to use different data
DATASET_URL = os.getenv(
    "AI_LEADER_DATASET_PATH",
    "https://docs.google.com/spreadsheets/d/e/2PACX-1vSU5zvx8wgk9FMEcRGlCtXkE4_T90OgsrqU4QPNZC478Rsp5JEBEEUjvlMkY3iMoiAmpa1zQ5QFkgT5/pub?output=csv",
)

## Step 2 — Load the evaluation dataset

The CSV holds labeled customer requests with **gold routing targets** (department, category, etc.). We use it to measure how the routing prototype performs beyond a handful of ad-hoc examples.

**Where the file comes from:** the path is set in the previous cell as `DATASET_URL`. By default that is a published Google Sheet CSV URL. To use a **local copy** instead (offline runs or if the URL changes), set the environment variable `AI_LEADER_DATASET_PATH` to a `.csv` file path before running the load cell — the same variable name is read in Settings.

If loading fails, the next cell raises a short error that repeats the URL or path you tried.

In [ ]:
df = load_and_validate_dataset(DATASET_URL)
print(f"Rows loaded: {len(df)}")
df.head(3)

**Optional:** inspect how departments are distributed in the reference labels.

In [ ]:
df["Routing to Department"].value_counts()

In [ ]:
df["Category"].value_counts()

If one department represents fewer than 10% of rows, accuracy scores will overstate overall performance — a model that never predicts that class can still score 90%+. In that case, check macro F1 alongside accuracy in Step 5. Department distribution also tells you where labeling effort is concentrated and where edge cases are likely sparse.

## Step 3 — Compare candidate models

Run all candidates on a random 30% sample of the dataset and compare them side by side.

These four models were shortlisted in Module 3.2 (Playground) based on output schema compliance and routing stability. Running them on a 30% sample keeps the comparison fast without losing signal on the main quality dimensions.

### What we measure here

| Metric | What it measures | Pass direction | Why it matters here |
|--------|-----------------|----------------|---------------------|
| **Department accuracy** | Share of tickets routed to the correct department | Higher | Primary routing KPI — maps to H2 |
| **Category accuracy** | Share of tickets with the correct category label | Higher | Supporting signal for H2 |
| **Unsafe auto-route rate** | Among High-confidence tickets, share routed to the wrong department | Lower | Safety guardrail — a high value means confidence is poorly calibrated |
| **Monthly cost (USD)** | Projected inference spend at 20,000 messages/month | Lower | Unit economics — maps to H6 |
| **p95 latency (ms)** | 95th-percentile model response time | Lower | Speed — maps to H7 |

**How the best model is chosen:** lowest misroute rate; ties broken by higher department accuracy, then lower cost, then lower latency. You can override the suggestion if you have a reason.



The system prompt defines everything the model knows about this task: the allowed categories, the allowed departments, the required output format, and the confidence signal. Print it below to confirm the structure before running comparisons. If any label names here differ from the dataset column values, matches will fail silently.

In [ ]:
print(DEFAULT_SYSTEM_PROMPT)

In [ ]:
CANDIDATE_MODELS = [
    "deepseek-ai/DeepSeek-V4-Pro",
    "zai-org/GLM-5.2",
    "openai/gpt-oss-120b",
    "Qwen/Qwen3.5-397B-A17B",
]

model_runs = await run_model_comparison_async(
    df=df.sample(frac=0.3, random_state=42),
    models=CANDIDATE_MODELS,
    api_key=api_key,
    system_prompt=DEFAULT_SYSTEM_PROMPT,
)

In [ ]:
comparison_df = build_model_comparison_dataframe(df, model_runs)
comparison_df

**Optional charts:** quality vs cost (and safety bar chart when there are 2+ models). Figures use `show_figure` so they render once in `inline` backends.

In [ ]:
if not comparison_df.empty:
    show_figure(plot_quality_vs_cost(comparison_df))
    show_figure(plot_safety_comparison(comparison_df))

These charts compress the comparison table into two views.

**Quality vs cost:** a model in the upper-left — high accuracy, low cost — is the strongest candidate. A model that is more accurate but significantly more expensive only makes sense if the accuracy gap is large enough to justify the difference at 20,000 messages/month.

**Safety comparison:** look at the gap between auto-route coverage and auto-route precision. A model that covers many cases but with low precision is overconfident — it will produce wrong auto-routes at scale. Prefer a model with a smaller, more accurate coverage over a wide but unreliable one.

If no model clearly dominates both charts, apply the selection rule: remove models that fail the routing quality threshold, then choose the cheapest among the remaining.

## Step 4 — Run full evaluation on the selected model

Evaluate every row in the dataset with the model you chose. The results feed all four hypothesis checks in Steps 5–8.

We use the same prompt as the comparison run (`DEFAULT_SYSTEM_PROMPT`) so Step 5–8 results are directly comparable to the model shortlist.

`select_best_model` applies this rule: lowest misroute rate first; ties broken by higher department accuracy, then lower cost, then lower latency. If you disagree with the suggestion — for example, if two models are within 2% accuracy but differ significantly on cost — override `BEST_MODEL` in the next cell before running.

In [ ]:
# Override here if you prefer a different candidate — see explanation above
BEST_MODEL = select_best_model(model_runs)
print("Suggested model:", BEST_MODEL)

In [ ]:
# Full evaluation uses the same prompt as the Step 3 comparison.
# Printed here for reference.
print(DEFAULT_SYSTEM_PROMPT)

In [ ]:
client = create_client(api_key, model=BEST_MODEL)

selected_run = await evaluate_model_on_dataframe_async(
    df=df,
    model=BEST_MODEL,
    client=client,
    system_prompt=DEFAULT_SYSTEM_PROMPT,
)

## Step 5 — Check routing quality

How often does the model route to the correct department and assign the correct category?

Pass threshold: **≥85% department accuracy** and **misroute rate ≤10%**.

These two metrics are the primary check for H2. Use category accuracy and F1 as supporting diagnostics — they help identify which labels the model confuses, but the routing decision is based on department accuracy and misroute rate.

In [ ]:
display_quality_metrics(quality_metrics=selected_run.quality_metrics)

Department accuracy and misroute rate determine whether H2 passes at this stage.

- A misroute rate above 10% means the baseline prompt is not ready for the next step — but this is expected at this stage and addressed in Notebook B.
- Check the confusion matrix: if most errors fall between two specific departments (e.g. Returns vs Customer Support), the prompt likely needs clearer label definitions for those classes.
- If accuracy is close to threshold, look at which cases are failing before deciding to proceed or iterate.

## Step 6 — Check safety

**Simple policy:** tickets where the model returns High confidence are candidates for auto-routing; everything else goes to human review.

This is not full safety validation — it is a practical MVP guardrail. The question it answers: if we auto-route only High-confidence cases, how often do we get it wrong?

Pass threshold: **unsafe auto-route rate ≤3%**.

In [ ]:
sm = compute_safety_metrics(df, selected_run.predictions)
display_safety_metrics(safety_metrics=sm)

The unsafe auto-route rate is the key number: it tells you how often the model would confidently send a ticket to the wrong department without human review.

- An unsafe auto-route rate above 10% means confidence is poorly calibrated — the model says "High" but is frequently wrong. Auto-routing is not safe in this state.
- Compare auto-route precision against overall department accuracy. If the gap is small, confidence is not adding useful signal and the guardrail offers little protection.
- High auto-route coverage paired with a high error rate is the most dangerous combination: the model is overconfident and wrong at scale.

If the unsafe auto-route rate fails the threshold, do not expand automation in the current workflow. Address calibration in Notebook B before proceeding.


## Step 7 — Check cost

How much would it cost to run this model at Candlekeep's volume of 20,000 messages/month?

Pass threshold: **<$1,000/month**.

For routing tasks, inference is usually not the primary cost risk — the model input and output are short. The more relevant question is whether a more expensive model delivers enough accuracy improvement to justify the difference in monthly spend.


In [ ]:
display_cost_metrics(cost_metrics=selected_run.cost_metrics)

The tables below break down cost per message and project monthly/annual totals.

In [ ]:
display_cost_breakdown(
    model=BEST_MODEL,
    cost_metrics=selected_run.cost_metrics,
)

In [ ]:
display_cost_projection(
    cost_metrics=selected_run.cost_metrics,
)

Cost per message and monthly projection are the two numbers that matter here.

- If projected monthly cost is well below $1,000, cost is not the constraint — focus attention on quality and safety thresholds instead.
- If a more expensive model passed quality thresholds in Step 5 with a meaningfully higher accuracy, compare the cost difference against the routing improvement. A $50/month gap between two models is rarely the deciding factor; a $500/month gap might be.
- The annual projection helps frame the cost in business terms when presenting to stakeholders.

## Step 8 — Check speed

How fast does the model respond? This step measures **LLM inference latency** — the time from API call to response. This is not the same as end-to-end response time in a live workflow, which also includes retrieval, post-processing, and delivery.

Pass threshold: **p95 latency <5,000 ms**.


In [ ]:
display_latency_metrics(latency_metrics=selected_run.latency_metrics)

Latency matters differently depending on where routing sits in the workflow.

- For customer-facing acknowledgments, p95 above 5,000 ms will create noticeable delays. For internal routing that runs before a specialist picks up the ticket, the threshold is more relaxed.
- If p95 exceeds 5,000 ms, a smaller or faster model is worth testing in Notebook B — most routing tasks do not require a large model, and model size is the main driver of inference latency.
- Median latency is useful for typical-case planning; p95 latency is what matters for user experience under load.

## Step 9 — MVP decision

The decision table combines all four dimensions — routing quality, safety, cost, and speed — into a single pass/fail summary against the thresholds set in the intro. This is the evidence you carry into a go/no-go conversation.

A single failing dimension does not mean stop. It means: identify whether the failure is fixable (prompt improvement, data quality, model swap) or structural (the task is too complex for this approach).

In [ ]:
decision = evaluate_decision(run=selected_run)
display_mvp_decision(decision=decision)

## Summary

This notebook produced a baseline: routing quality, safety, cost, and latency measured before any optimization.

Failing one or more thresholds at this stage is expected — the baseline prompt is intentionally simple. The usual outcome is **Improve and re-test** in Notebook B. A hard stop is only warranted when a constraint clearly cannot be closed by prompt improvement, model swap, or data quality work.

**Record your baseline results here before moving to Notebook B:**

| Dimension | Your result | Pass threshold | Status |
|-----------|-------------|----------------|--------|
| Department accuracy | | ≥85% | |
| Misroute rate | | ≤10% | |
| Unsafe auto-route rate | | ≤3% | |
| Monthly cost | | <$1,000 | |
| p95 latency | | <5,000 ms | |

**Notebook B focuses on:**
- Refining the prompt based on observed mistakes
- Inspecting high-confidence errors for calibration issues
- Improving label quality for ambiguous cases
- Testing few-shot examples and analyzing quality vs cost/latency trade-offs

This notebook defines the gap. Notebook B is where you close it.
